# 03. LightGBM 학습 및 예측

## 🎯 목적

02단계에서 생성한 통합 데이터셋으로 **전체 통합 LightGBM 모델** 학습

## 📋 핵심 설계

### ✅ Target: 로그 종가 절대값
```python
target_log_close = log(close)
```
- t일 features → t+1일 log_close 예측
- 절대값 예측이므로 종목 간 가격 수준 차이 반영

### ✅ Feature: ticker 제외
- **제외 이유**: 신규 종목 예측 불가, 차원 폭발
- **대체 수단**: Meta features (liquidity_score, risk_composite)
- **장점**: 2,900개 → 12개 features로 일반화

### ✅ 학습 방식: 전체 통합 모델
- 단일 모델로 모든 종목 처리
- 종목 간 공통 패턴 학습
- 신규 상장 종목도 즉시 예측 가능

## 🔄 데이터 흐름

```
data/03_processed/dataset.parquet
    ↓ [load]
  + Target 생성: log(close)
    ↓ [walk_forward_split]
  train / valid (4 folds) / test
    ↓ [LightGBM 학습]
  Unified Model (NO ticker feature)
    ↓ [save]
data/04_models/*.pkl
data/05_results/*.csv
```

## 🔧 Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings

# 모듈 임포트
from src.utils.config import load_config
from src.models.lightgbm_model import LightGBMModel
from src.modeling.trainer import WalkForwardTrainer
from src.models.artifact import save_model_artifact

warnings.filterwarnings('ignore')

In [ ]:
# 1. 설정 로드
cfg = load_config()
ref_date = cfg['project']['reference_date']
train_cfg = cfg['training']

# 2. 경로 설정 (Input: 02_processed / Output: 03_results / Model: 04_models)
input_dir = Path(cfg['paths']['processed_dir']) / ref_date
result_base_dir = Path(cfg['paths']['result_dir']) / ref_date
model_dir = Path(cfg['paths']['model_dir']) / ref_date
csv_pred_dir = result_base_dir / "csv"

# 디렉토리 생성
model_dir.mkdir(parents=True, exist_ok=True)
csv_pred_dir.mkdir(parents=True, exist_ok=True)

print(f"🚀 [Step 3] 학습 및 예측 시작 (기준일: {ref_date})")
print(f"   📂 Input: {input_dir}")
print(f"   📂 Output: {result_base_dir}")

## 1️⃣ 데이터 로드 및 Target 생성

In [ ]:
print("\n📥 Loading dataset & Creating Target...")

# 데이터 로드 및 정렬
df = pd.read_parquet(input_dir / "dataset.parquet")
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

target_col = train_cfg['target_col']
feature_cols = [c for c in df.columns if c.startswith('feature_') or c in ['liquidity_score', 'risk_composite']]

# 1. Target 생성: 현재 행(t)의 로그 종가
df[target_col] = np.log(df['close'])

# 2. Feature Shift: t일 행에 t-1일의 피처를 배치 (의도: 어제 정보로 오늘 종가 예측)
print(f"   🔄 Shifting {len(feature_cols)} features...")
for col in feature_cols:
    df[col] = df.groupby('ticker')[col].shift(1)

# 3. 결측치 제거: shift(1)로 발생한 각 종목의 첫 행(NaN) 제거
df_train = df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

print(f"   - 학습 가능 행수: {len(df_train):,}")

## 2️⃣ 모델 초기화 & Trainer 실행

In [ ]:
print("\n🔧 Initializing Model...")

model = LightGBMModel(
    model_version=f"v1_unified_{ref_date}",
    params=train_cfg['lgbm_params'],
    feature_list=feature_cols,
    categorical_features=[], # Ticker를 Feature로 쓰지 않음 (General Model)
    task="regression"
)

trainer = WalkForwardTrainer(
    model=model,
    feature_cols=feature_cols,
    target_col=target_col,
    date_col='date'
)

## 3️⃣ Walk-Forward 학습 실행

In [ ]:
print("\n🏃 Running Walk-Forward Training...")

results = trainer.run(
    df=df_train,
    train_end=train_cfg['train_end'],
    valid_window_days=train_cfg['valid_window_days'],
    test_window_days=train_cfg['test_window_days'],
    num_valid=train_cfg['num_valid'],
    fit_kwargs={
        'num_boost_round': 1000,
        'callbacks': [
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(0) # 로그 간소화
        ]
    }
)

## 4️⃣ 성능 평가

In [ ]:
print("\n" + "="*60)
print("📊 Performance Evaluation (RMS Aggregated)")
print("="*60)

# Helper: RMS 계산 함수
def calc_rms(values):
    return np.sqrt(np.mean(np.array(values)**2))

# 1. Validation Score 집계 (RMS 적용)
valid_metrics_list = results['valid_metrics'] # List of dicts
valid_summary = {}

if valid_metrics_list:
    metric_keys = valid_metrics_list[0].keys()
    for key in metric_keys:
        if key == 'samples':
            valid_summary[key] = sum(d[key] for d in valid_metrics_list)
        else:
            # 사용자의 요청: RMSE, MAE 등을 RMS 방식으로 평균
            values = [d[key] for d in valid_metrics_list]
            valid_summary[key] = calc_rms(values)

    print(f"\n[Validation Set - {train_cfg['num_valid']} Folds Aggregated]")
    for k, v in valid_summary.items():
        print(f"  {k}: {v:.6f}")
else:
    print("\n[Validation Set] No validation performed.")

# 2. Test Score
print(f"\n[Test Set]")
for k, v in results['test_metrics'].items():
    print(f"  {k}: {v:.6f}")

# 3. Overfitting Check
if valid_metrics_list:
    gap = results['test_metrics']['rmse'] - valid_summary['rmse']
    print(f"\n[Overfitting Check]")
    print(f"  Gap (Test - Valid RMS): {gap:+.6f}")
    print("  ✅ Stable" if gap < 0.02 else "  ⚠️ Warning: Potential Overfitting")

## 5️⃣ 모델 및 결과 저장

In [ ]:
# ==================== 모델 저장 ====================

print("\n💾 Saving model artifact...")

final_model = results.get('final_model', model)
artifact_path = save_model_artifact(
    model_name="lightgbm",
    model_version=f"v1_rolling_{ref_date}",
    model_object=final_model,
    metadata={
        "train_end": train_cfg['train_end'],
        "test_rmse": results['test_metrics']['rmse'],
        "valid_rmse_rms": valid_summary.get('rmse')
    },
    model_dir=model_dir  # 변경된 경로 적용
)
print(f"   - [Model] Saved to: {model_dir}")

In [ ]:
# ==================== 예측 결과 저장 ====================

print("\n💾 Saving predictions...")
from tqdm import tqdm

pred_df = results['test_predictions']
pred_df = pred_df.merge(df[['date', 'ticker', 'close']], on=['date', 'ticker'], how='left')
pred_df['pred_price'] = np.exp(pred_df['y_pred'])
pred_df = pred_df.rename(columns={'close': 'real_price'})

# A. 통합 Parquet 저장 (파이프라인용)
full_parquet_path = result_base_dir / "predictions.parquet"
pred_df.to_parquet(full_parquet_path, index=False)
print(f"   - [Total] Saved Parquet: {full_parquet_path}")

# B. 종목별 개별 CSV 저장 (디버깅/사람용)
# ticker_name_map이 있다면 사용 (01단계에서 생성한 master 활용 권장)
print(f"   - [Individual] Saving ticker CSVs to {csv_pred_dir}...")

# ticker_name_map을 로드하여 한글 종목명 파일명 적용
try:
    master_path = Path(cfg['paths']['raw_dir']) / ref_date / f"ticker_master_{ref_date}.csv"
    df_master = pd.read_csv(master_path)
    ticker_name_map = dict(zip(df_master['ticker'].astype(str), df_master['name']))
except Exception:
    ticker_name_map = {}

for ticker, group in tqdm(pred_df.groupby('ticker'), desc="Saving CSVs"):
    name = ticker_name_map.get(str(ticker), f"ticker_{ticker}")
    safe_name = str(name).replace('/', '_').replace('\\', '_')
    group.to_csv(csv_pred_dir / f"{safe_name}.csv", index=False, encoding='utf-8-sig')

In [ ]:
# ==================== 기타 리포트 저장 ====================

print("\n💾 Saving feature importance & metrics...")

# 중요도 저장
imp_df = pd.DataFrame(
    final_model.get_meta().get('feature_importance', {}).items(), 
    columns=['feature', 'importance']
).sort_values('importance', ascending=False)
imp_df.to_csv(result_base_dir / "feature_importance.csv", index=False)

# 지표 요약 저장
metrics_summary = pd.DataFrame([{
    'reference_date': ref_date,
    'test_rmse': results['test_metrics']['rmse'],
    'valid_rmse_rms': valid_summary.get('rmse')
}])
metrics_summary.to_csv(result_base_dir / "metrics_summary.csv", index=False)

print(f"\n✅ [Step 3] 모든 산출물 저장 완료")

## ✅ 완료!

### 생성된 파일
- 모델: `data/04_models/lightgbm/*.pkl`
- 예측: `data/05_results/predictions_*.csv`
- 지표: `data/05_results/metrics_summary_*.csv`
- Feature Importance: `data/05_results/feature_importance_*.csv`

### 핵심 설계 특징

✅ **Ticker를 feature로 사용하지 않음**
- 신규 종목 예측 가능
- 차원 폭발 문제 없음
- Meta features (liquidity, risk)로 종목 특성 표현

✅ **로그 종가 절대값 예측**
- t일 features → t+1일 log(close)
- 종목 간 가격 수준 차이 반영

✅ **전체 통합 모델**
- 단일 모델로 모든 종목 처리
- 종목 간 공통 패턴 학습

### 모델 사용 예시
```python
from src.models.lightgbm_model import LightGBMModel

# 신규 종목 데이터 준비
new_stock = pd.DataFrame({
    'feature_ma_5': [...],
    'liquidity_score': [...],  # Meta feature
    'risk_composite': [...]     # Meta feature
    # ticker 필요 없음!
})

# 예측
loaded_model = LightGBMModel.load('data/04_models/lightgbm/*.pkl')
predictions = loaded_model.predict(new_stock)
```

### 다음 단계
- **트랙 D**: Universe 선정 및 시그널 생성
- **백테스트**: 실제 매매 시뮬레이션